# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

In [1]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"  # твоя публичная ссылка
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"]  # это уже прямая ссылка на файл

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


In [2]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [3]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [4]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


Здесь 1.0 соответствует оценке RELEVANT_PLUS, 0.1 -- оценке RELEVANT_MINUS, 0.0 -- оценке IRRELEVANT.

Ваша задача -- построить LLM-агента, который будет предсказывать релевантность.

Выделим данные для оценки качества агента. Запуск агента -- это тяжелая и потенциально дорогая операция. Поэтому eval-множество имеет размер 500. Также для простоты из eval-множества выкинуты данные с оценкой RELEVANT_MINUS. Тем не менее, вы можете использовать такие примеры для подачи примеров агенту.

**ОБРАТИТЕ ВНИМАНИЕ, ЧТО В EVAL-ДАННЫЕ НЕЛЬЗЯ ПОДГЛЯДЫВАТЬ ДЛЯ КАЛИБРОВКИ АГЕНТА!!! ДЛЯ ЭТОГО ЕСТЬ ОБУЧАЮЩИЕ ДАННЫЕ**

В качестве метрики качества мы будем использовать обычную ACCURACY, поскольку классы сбалансированы.

In [5]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [6]:
eval_data.to_excel("eval_data.xlsx")

# Агент


## Агент с тулзой

In [7]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [8]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool
import os
import time, uuid
from dotenv import load_dotenv
load_dotenv()

True

## Агент без вызова тулзов


In [137]:
open_router_model_name = "xiaomi/mimo-v2-flash:free"

llm = ChatOpenAI(
    model=open_router_model_name,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.2,
    max_tokens=512,
    timeout=60,
    max_retries=3)

In [138]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_raw = TavilySearchResults(
    max_results=10,
    include_answer=True,
    include_raw_content=False,
)

TAVILY_LOG_PATH = "tavily_log.jsonl"

def _log_jsonl(obj: dict, path: str = TAVILY_LOG_PATH):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def tavily_logged(query: str) -> dict:
    run_id = str(uuid.uuid4())
    t0 = time.time()
    _log_jsonl({"type": "tool_start", "run_id": run_id, "tool": "tavily", "query": query, "ts": t0})

    out = tavily_raw.invoke({"query": query})

    dt = time.time() - t0
    answer = out.get("answer", "") if isinstance(out, dict) else ""
    results = out.get("results", []) if isinstance(out, dict) else []

    _log_jsonl({
        "type": "tool_end",
        "run_id": run_id,
        "tool": "tavily",
        "query": query,
        "latency_sec": dt,
        "answer": answer,
        "top_results": results[:3] if isinstance(results, list) else [],
    })
    return out


In [299]:
train_few_shot_ids = [571, 571, 586, 581, 590, 983, 580]


PLAN_SYSTEM_PROMPT = f"""
Ты - ассистент, который оценивает, строгую релевантность организаций на картах широким рубричным запросам.\n
Примеры рубричных запросов: "ресторан с верандой", "романтичный джаз-бар".\n
ВАЖНО : В рубричном запросе очень важны детали, если деталь не верна, то запрос не релевантен.
ВАЖНО : Если во входных данных нет информации о какой-то детали запроса,
то ее нужно уточнить через needs_search=true и search_queries

Примеры входных данных:\n
{make_example(train_data.loc[586],is_train=True)}\n\n
{make_example(train_data.loc[580],is_train=True)}\n\n

ОЧЕНЬ ВАЖНО: Формат ответа (строго):
{{
  "version": "1.0",
  "label": null,
  "confidence": <float 0..1>,
  "needs_search": <true/false>,
  "missing_details": [<строки>],
  "search_queries": [<строки>],
  "search_results": [],
  "rationale": "<1-2 предложения>",
  "decision_rules_hit": [<строки>]
}}

Правила:
- Ты НЕ выдаёшь финальный label. label ВСЕГДА null.
- Если confidence >= 0.80: needs_search=false и search_queries=[].
- Если confidence < 0.80: needs_search=true и search_queries содержит 1-2 запроса.
- search_queries должны проверять КОНКРЕТНЫЕ отсутствующие детали.
- При составлении запросов на каждую деталь 1-2 запроса.
- Шаблон query: "Официальный сайт <ПОЛНОЕ НАЗВАНИЕ ОРГАНИЗАЦИИ> <ОТСУТСТВУЯЩАЯ ДЕТАЛЬ>"
- ПРИМЕР query: "Официальный сайт Ингосстрах застраховать автомобиль"
- Шаблон query: "<ОТСУТСТВУЯЩАЯ ДЕТАЛЬ> в <ПОЛНОЕ НАЗВАНИЕ ОРГАНИЗАЦИИ> <ГОРОД, АДРЕС>"
- Пример query: "Страхование автомобиля в Ингосстрах Вологодская область, рабочий посёлок Чагода, улица Кирова, 5Б

- Ты возвращаешь СТРОГО ТОЛЬКО JSON, ЗАПРЕЩЕНО использовать Markdown, тройные кавычки и блоки ```json.
- Первый символ ответа ДОЛЖЕН быть '{' и последний символ ответа ДОЛЖЕН быть '}'.
- Если ты выводишь что-то кроме JSON — это ошибка.

СЕЙЧАС: проанализируй вход и верни JSON.
"""

DECIDE_SYSTEM_PROMPT = f"""
Ты — ассистент, который принимает:
1) исходный запрос + карточку организации
2) результаты веб-поиска (search_results)
и возвращает финальное решение о строгой релевантности.

Примеры входных данных:\n
{make_example(train_data.loc[570],is_train=True)}\n\n
{make_example(train_data.loc[571],is_train=True)}\n\n
{make_example(train_data.loc[586],is_train=True)}\n\n
{make_example(train_data.loc[581],is_train=True)}\n\n
{make_example(train_data.loc[590],is_train=True)}\n\n
{make_example(train_data.loc[983],is_train=True)}\n\n

Ты ВСЕГДА отвечаешь ТОЛЬКО валидным JSON-объектом. Никакого текста вне JSON.

Формат ответа (строго):
{{
  "version": "1.0",
  "label": <0 или 1>,
  "confidence": <float 0..1>,
  "needs_search": false,
  "missing_details": [<строки>],
  "search_queries": [<строки>],
  "search_results": [<объекты>],
  "rationale": "<1-3 предложения>",
  "decision_rules_hit": [<строки>]
}}

Правила принятия решения:
- Для каждого ответа из search_results оцени его релевантность:
изучи 'search_results':'answer', если в нем нет упоминания конкретной огранизации с ПОЛНЫМ названием,
то этот ответ из search_results не считается релевантным
и НЕ ДОЛЖЕН использоваться при принятии решений о релевантности.
- Если search_results пустой или нерелевантный и деталь не подтверждена → label=0 (строго).
- Если обязательная деталь запроса НЕ подтверждается (в карточке + search_results) → label=0.
- Если обязательные детали подтверждаются и нет противоречий → label=1.

СЕЙЧАС: используй search_results и верни финальный JSON.
"""


In [300]:
def parse_json_safe(text: str) -> dict:
    """
    Парсит JSON, даже если модель вернула JSON дважды закодированный (строка внутри строки).
    """
    if (text[0] + text[-1]) not in '{}':
      text = text[text.find('{'):]
      text = text[:text.find("```")]
    obj = json.loads(text)
    if isinstance(obj, str):
        obj = json.loads(obj)
    if not isinstance(obj, dict):
        raise ValueError(f"Expected dict JSON, got: {type(obj)}")
    return obj

from typing import Any, Dict, List

def normalize_tavily_results(tavily_out: Any, query: str, score_threshold: float = 0.55, top_k: int = 5) -> dict:
    """
    Нормализует выход Tavily к единому формату.

    Поддерживаемые форматы входа:
    1) dict: {"answer": str, "results": [ {title,url,content,score,...}, ... ]}
    2) list: [ {title,url,content,score,...}, ... ]  (как в твоём примере)

    Возвращает:
    {
      "query": str,
      "answer": str,
      "top_results": [ {title,url,content,score}, ... ]
    }
    """
    answer = ""
    results: List[Dict[str, Any]] = []

    # формат 1: dict
    if isinstance(tavily_out, dict):
        answer = tavily_out.get("answer") or ""
        raw_results = tavily_out.get("results", [])
        if isinstance(raw_results, list):
            results = [r for r in raw_results if isinstance(r, dict)]

    # формат 2: list
    elif isinstance(tavily_out, list):
        results = [r for r in tavily_out if isinstance(r, dict)]

    # неизвестный формат
    else:
        results = []

    filtered = []
    for r in results:
        score = r.get("score", None)
        if isinstance(score, (int, float)) and score >= score_threshold:
            filtered.append({
                "title": r.get("title", "") or "",
                "url": r.get("url", "") or "",
                "content": r.get("content", "") or "",
                "score": float(score),
            })

    filtered.sort(key=lambda x: x["score"], reverse=True)
    filtered = filtered[:top_k]

    return {
        "query": query,
        "answer": answer,
        "top_results": filtered
    }


In [301]:
def run_relevance_agent(prompt_text: str,
                        score_threshold: float = 0.55,
                        max_search_queries: int = 2) -> dict:
    """
    1) PLAN: LLM решает, нужен ли поиск и какие queries
    2) SEARCH: вручную вызываем tavily_logged по queries
    3) DECIDE: LLM получает search_results и выдаёт финальный label
    Возвращает итоговый JSON dict (как у DECIDE_SYSTEM_PROMPT).
    """

    # ---- Step 1: PLAN ----
    plan_msg = llm.invoke([
        SystemMessage(content=PLAN_SYSTEM_PROMPT),
        HumanMessage(content=prompt_text),
    ])
    plan = parse_json_safe(plan_msg.content)
    # если модель считает, что поиск не нужен — можем сразу решить без поиска,
    # но по твоей задаче ты хочешь финальный label всегда на втором шаге.
    # Поэтому просто передадим пустые search_results.
    queries = plan.get("search_queries", []) or []
    needs_search = bool(plan.get("needs_search", False))

    search_results = []
    if needs_search and len(queries) > 0:
        for q in queries[:max_search_queries]:
            tav = tavily_logged(q)
            packed = normalize_tavily_results(tav, q, score_threshold=score_threshold, top_k=5)
            # добавляем даже если top_results пуст — модель увидит, что подтверждений нет
            search_results.append(packed)
    print('search_results : \n', search_results, '\n\n')
    # ---- Step 2: DECIDE ----
    decide_input = {
        "prompt": prompt_text,
        "plan": plan,
        "search_results": search_results
    }
    print('decide_input : \n', decide_input, '\n\n')
    decide_msg = llm.invoke([
        SystemMessage(content=DECIDE_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(decide_input, ensure_ascii=False)),
    ])
    final_obj = parse_json_safe(decide_msg.content)
    return final_obj


In [302]:

run_relevance_agent(prompt_text=prompt, score_threshold=0.3, max_search_queries=3)

KeyboardInterrupt: 

In [305]:
from random import random

start = 1000
n = 20
train_few_shot_ids = [570, 571, 586, 581, 590, 983]

outputs = []
pred = []
true = []

for i in range(start, start + n):
    if i in train_few_shot_ids:
        continue
    if float(train_data.loc[i]["relevance"]) == 0.1:
        continue
    if float(train_data.loc[i]["relevance"]) == 0.0 and random() < 0.8:
      continue

    prompt = make_example(train_data.loc[i])
    out = run_relevance_agent(prompt_text=prompt, score_threshold=0.2, max_search_queries=6)

    outputs.append({"idx": i, "out": out})
    pred.append(int(out["label"]))
    true.append(int(train_data.loc[i]["relevance"]))

    print(i, out["label"], out.get("confidence"))


search_results : 
 [{'query': 'Официальный сайт Киров, улица Менделеева, 2 такелажные работы', 'answer': '', 'top_results': [{'title': 'Кировская такелажная компания - Справочник Кирова - Spravker.ru', 'url': 'https://kirov.spravker.ru/arhivnoe-oborudovanie/takelazhnaya-kompaniya-kirovskaya.htm', 'content': '43 128\nорганизаций\n\n# Кировская такелажная компания\n\n ВКонтакте\n Одноклассники\n Мой Мир\n + Twitter\n  + WhatsApp\n  + Telegram\n  + Скопировать ссылку; "Скопировать ссылку")\n\nАдрес\n:   ул. Менделеева, 2, Киров\n    На\n    карте\n\nТелефон\n:   +7 (8332) 76-02-54, +7 (8332) 77-12-54, +7 (8332) 76-00-73, +7 (8332) 77-11-68\n\nРежим работы\n:   пн-пт 08:00–17:00\n\nСайт\n:   ktk43.ru\n\nСоциальные сети\n:   Telegram, WhatsApp, Вконтакте\n\n Нашли ошибку в описании?\n Я — владелец\n\n### Написать отзыв — Кировская такелажная компания\n\nИмя \\\n\nВаша оценка \\\n\nОценка формирует рейтинг компании\n\nВаш отзыв \\\n\nСогласен на обработку персональных данных\n\nОтзыв не долж

In [306]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

precision, recall, f1, _ = precision_recall_fscore_support(true, pred, average="binary", zero_division=0)
acc = accuracy_score(true, pred)

print("precision:", round(precision, 4))
print("recall   :", round(recall, 4))
print("f1       :", round(f1, 4))
print("accuracy :", round(acc, 4))


precision: 1.0
recall   : 0.5714
f1       : 0.7273
accuracy : 0.6667


In [303]:
prompt = make_example(train_data.loc[1025])
out = run_relevance_agent(prompt_text=prompt, score_threshold=0.2, max_search_queries=6)
print(out["label"], out.get("confidence"))

search_results : 
 [{'query': 'Официальный сайт Московская область, Истра, Рабочая улица, 2 массаж', 'answer': '', 'top_results': [{'title': 'Массаж на Рабочей улице (Истра)', 'url': 'https://zoon.ru/msk/gorod-istra/beauty/type-massazh/street-rabochaya_ulitsa/', 'content': 'Истра\n   Салоны красоты\n   Массаж\n   Рабочая улица\n\n# Массаж на Рабочей улице (Истра)\n\n Спины\n Массаж ног\n Массаж груди\n Массаж бедер\n Ламинирование бровей\n Карбокситерапия\n Rf-лифтинг\n Долговременная укладка бровей\n Коррекция фигуры\n Ботокс ресниц\n Сужение пор\n Фракционная мезотерапия\n Антицеллюлитное обертывание\n Макияж на выпускной\n Увеличение губ филлерами\n\nНашли для вас:\n5 мест и 6 неподалёку\n\n 5 массажных салонов на Рабочей улице в Истре – найдите лучшие с помощью Zoon;\n У нас актуальная и проверенная информация о заведениях, удобный поиск на карте Истры;\n Массаж на Рабочей улице – читайте отзывы реальных клиентов и записывайтесь.\n\n0\n\nУслуги на дому\n\nЛицо и тело\n\nМетро, райо